In [1]:
import numpy as np
import os

In [2]:
path = "/home/twlodarski/Projects/ribosome_tunnels/bacteria/vnat/"

In [3]:
os.chdir(path)

In [4]:
lengths = [10, 20, 30, 40, 60]

In [40]:
%%bash -s "{' '.join(map(str, lengths))}"
# The line above passes the Python 'lengths' list into the bash script as $1, $2, etc.
GMX_PATH="/home/twlodarski/soft/gromacs-2021/bin/gmx"

for item in $@; do
    if [ -d "$item" ]; then
        cd "$item"
        # 1. Generate Index silently
        # We look for residue 166 as the fitting anchor
        $GMX_PATH make_ndx -f NC_FME_${item}.pdb -o index_r.ndx > /dev/null 2>&1 << EOF
r 166
q
EOF
        # 2. Fix PBC (nojump) silently
        $GMX_PATH trjconv -f md_run.xtc -s NC_FME_${item}.pdb -pbc nojump -o traj_nojump.xtc > /dev/null 2>&1 << EOF
0
EOF
        # 3. Fitting (Rot+Trans) silently
        # Group 10 (r 166) is the reference, Group 0 (System) is the output
        # Group 14 (r 166) is the reference in bacteria
        $GMX_PATH trjconv -s NC_FME_${item}.pdb -f traj_nojump.xtc -n index_r.ndx -o fitted.xtc -fit rot+trans > /dev/null 2>&1 << EOF
14
0
EOF
        echo "✅ Processed length $item: fitted.xtc created."
        cd ../
    else
        echo "⚠️  Directory $item not found, skipping..."
    fi
done

✅ Processed length 10: fitted.xtc created.
✅ Processed length 20: fitted.xtc created.
✅ Processed length 30: fitted.xtc created.
✅ Processed length 40: fitted.xtc created.
✅ Processed length 60: fitted.xtc created.


In [24]:
import MDAnalysis as mda
import numpy as np
import os

ids = ['10', '20', '30', '40', '60']

for idx in ids:
    print(f"--- Analyzing Length: {idx} ---")    
    # Load the NC-only universe
    # Note: We use the PDB that matches the atom count of the fitted XTC
    u = mda.Universe(f"{idx}/NC_FME_{idx}.pdb", f"{idx}/fitted_{idx}.xtc")
    ca = u.select_atoms('name CA')
    all_atoms = u.atoms
    n_frames = len(u.trajectory)
    # 1. Get Reference 1 (Frame 0)
    u.trajectory[0]
    ref1_pos = ca.positions.copy()
    # 2. Find the frame furthest from Frame 0
    max_rmsd1 = -1
    frame1_idx = 0
    for ts in u.trajectory:
        # Calculate MSD manually for speed
        msd = np.mean(np.sum((ca.positions - ref1_pos)**2, axis=1))
        if msd > max_rmsd1:
            max_rmsd1 = msd
            frame1_idx = ts.frame
    print(f"  Frame {frame1_idx} found as first extreme (RMSD: {np.sqrt(max_rmsd1):.2f} Å)")
    # Save first reference
    u.trajectory[frame1_idx]
    ref2_pos = ca.positions.copy()
    all_atoms.write(f"{idx}/first_ref.pdb")
    # 3. Find the frame furthest from BOTH Frame 0 and Frame 1
    max_dist_sum = -1
    frame2_idx = 0
    for ts in u.trajectory:
        d1 = np.mean(np.sum((ca.positions - ref1_pos)**2, axis=1))
        d2 = np.mean(np.sum((ca.positions - ref2_pos)**2, axis=1))
        # Maximize the sum of distances to ensure we find a new conformational "corner"
        combined_dist = d1 + d2
        if combined_dist > max_dist_sum:
            max_dist_sum = combined_dist
            frame2_idx = ts.frame
    print(f"  Frame {frame2_idx} found as second extreme.")
    # Save second reference
    u.trajectory[frame2_idx]
    all_atoms.write(f"{idx}/second_ref.pdb")

--- Analyzing Length: 10 ---
  Frame 47582 found as first extreme (RMSD: 9.90 Å)


/home/twlodarski/anaconda3/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1154: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn("Found no information for attr: '{}'"


  Frame 25918 found as second extreme.
--- Analyzing Length: 20 ---
  Frame 43994 found as first extreme (RMSD: 19.04 Å)
  Frame 19814 found as second extreme.
--- Analyzing Length: 30 ---
  Frame 20648 found as first extreme (RMSD: 24.49 Å)
  Frame 6674 found as second extreme.
--- Analyzing Length: 40 ---
  Frame 16114 found as first extreme (RMSD: 30.60 Å)
  Frame 38827 found as second extreme.
--- Analyzing Length: 60 ---
  Frame 23872 found as first extreme (RMSD: 40.60 Å)
  Frame 16572 found as second extreme.


## Trajectory analysis for the ref_1 and ref_2 simulations

In [20]:
import numpy as np
import os

In [33]:
path = "/home/twlodarski/Projects/ribosome_tunnels/bacteria/vnat/"

In [34]:
os.chdir(path)

In [35]:
lengths = [10, 20, 30, 40, 60]

In [36]:
%%bash -s "{' '.join(map(str, lengths))}"
# The line above passes the Python 'lengths' list into the bash script as $1, $2, etc.
GMX_PATH="/home/twlodarski/soft/gromacs-2021/bin/gmx"

for item in $@; do
    if [ -d "$item" ]; then
        cd "$item""/ref_1"
        # 1. Generate Index silently
        # We look for residue 166 as the fitting anchor
        $GMX_PATH make_ndx -f ../NC_MET_${item}.pdb -o index_r.ndx > /dev/null 2>&1 << EOF
r 166
q
EOF
        # 2. Fix PBC (nojump) silently
        $GMX_PATH trjconv -f md_run.xtc -s ../first_ref.pdb -pbc nojump -o traj_nojump.xtc > /dev/null 2>&1 << EOF
0
EOF
        # 3. Fitting (Rot+Trans) silently
        # Group 10 (r 166) is the reference, Group 0 (System) is the output
        # Group 14 (r 166) is the reference in bacteria
        $GMX_PATH trjconv -s ../first_ref.pdb -f traj_nojump.xtc -n index_r.ndx -o fitted.xtc -fit rot+trans > /dev/null 2>&1 << EOF
14
0
EOF
        echo "✅ Processed length $item: fitted.xtc created."
        cd ../../
    else
        echo "⚠️  Directory $item not found, skipping..."
    fi
done

✅ Processed length 10: fitted.xtc created.
✅ Processed length 20: fitted.xtc created.
✅ Processed length 30: fitted.xtc created.
✅ Processed length 40: fitted.xtc created.
✅ Processed length 60: fitted.xtc created.


In [37]:
%%bash -s "{' '.join(map(str, lengths))}"
# The line above passes the Python 'lengths' list into the bash script as $1, $2, etc.
GMX_PATH="/home/twlodarski/soft/gromacs-2021/bin/gmx"

for item in $@; do
    if [ -d "$item" ]; then
        cd "$item""/ref_2"
        # 1. Generate Index silently
        # We look for residue 166 as the fitting anchor
        $GMX_PATH make_ndx -f ../NC_FME_${item}.pdb -o index_r.ndx > /dev/null 2>&1 << EOF
r 166
q
EOF
        # 2. Fix PBC (nojump) silently
        $GMX_PATH trjconv -f md_run.xtc -s ../second_ref.pdb -pbc nojump -o traj_nojump.xtc > /dev/null 2>&1 << EOF
0
EOF
        # 3. Fitting (Rot+Trans) silently
        # Group 10 (r 166) is the reference, Group 0 (System) is the output
        # Group 14 (r 166) is the reference in bacteria
        $GMX_PATH trjconv -s ../second_ref.pdb -f traj_nojump.xtc -n index_r.ndx -o fitted.xtc -fit rot+trans > /dev/null 2>&1 << EOF
14
0
EOF
        echo "✅ Processed length $item: fitted.xtc created."
        cd ../../
    else
        echo "⚠️  Directory $item not found, skipping..."
    fi
done

✅ Processed length 10: fitted.xtc created.
✅ Processed length 20: fitted.xtc created.
✅ Processed length 30: fitted.xtc created.
✅ Processed length 40: fitted.xtc created.
✅ Processed length 60: fitted.xtc created.
